# Assignment 03: Project AeroBrain

**Course:** Agentic AI: From Concepts to Practice

**Instructor:** Prof. Karthik Vaidhyanathan

**TAs:** Aviral Gupta, Aneetta Sara Shany, Ch Pavan Harshit, Shreyash Chandak

**Topic:** Retrieval-Augmented Generation (RAG)

**Deadline:** 9th August 2026, 11:59 PM

---

## Student Details

**Name:** Ravikumar M V

**Roll Number:** cert-aai-2026-06-0043

**Email:** ravikumarmv@gmail.com

---

## Instructions

- Run the notebook from top to bottom.
- Complete every cell marked **TODO**.
- Answer every reflection question in the markdown cell provided.
- Do **not** hardcode your Gemini API key. Use Colab Secrets.
- You will reuse your Assignment 02 chatbot in Part 5, so keep that notebook handy.
- Submit `AeroBrain_YourName.zip` containing this notebook and your screenshots.


---

# Setup

Run these four cells before anything else.


In [2]:
# Install required packages
# Run this cell first - it may take a minute or two.

!pip install flask flask-cors google-generativeai pypdf faiss-cpu --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 63.1 MB/s eta 0:00:00


In [4]:
# Imports

import os
import re
import time
import json
import random
import textwrap

import numpy as np
import faiss
from pypdf import PdfReader

from flask import Flask, request, jsonify
from flask_cors import CORS
from google.colab.output import eval_js
import google.generativeai as genai

print('All libraries imported successfully.')


All libraries imported successfully.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [5]:
# Read your Gemini API key from Colab Secrets
#
# Steps:
#   1. Click the key icon in the left sidebar
#   2. Add a secret named GEMINI_API_KEY and paste your key as the value
#   3. Toggle 'Notebook access' ON

from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

print('API key loaded.' if GEMINI_API_KEY else 'WARNING: API key not found.')


API key loaded.


In [6]:
# Model configuration
#
# GEN_MODEL       - the chat model, same one you used in Assignment 02
# EMBEDDING_MODEL - turns text into vectors. This is a different model from the
#                   chat model: it does not generate text, it only measures meaning.
#
# If the embedding model name below is rejected, run the loop underneath to see
# which embedding models your key currently has access to, and use one of those.

GEN_MODEL = 'gemini-3.1-flash-lite' # TODO: Add the model name
EMBEDDING_MODEL = 'gemini-embedding-2' # TODO: Add the embedding model name - pick one from the list printed below

gemini_model = genai.GenerativeModel(GEN_MODEL)

print('Embedding models available to you:')
for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print('  ', m.name)


Embedding models available to you:
   models/gemini-embedding-001
   models/gemini-embedding-2-preview
   models/gemini-embedding-2


---

# Part 1 - Building the Knowledge Base

## Objective

Turn a large technical PDF into small pieces that a language model can work with.

A language model can only read what fits in its context window. Our first job is to
break the manual into pieces small enough to send, but large enough to still make
sense on their own.


In [7]:
# Download the official Airbus A320 Airport Planning Manual.

MANUAL_URL = 'https://www.aircraft.airbus.com/sites/g/files/jlcbta126/files/2025-01/AC_A320_0624.pdf' # TODO: Add the URL
PDF_PATH = 'a320_manual.pdf'

!wget -q -O {PDF_PATH} {MANUAL_URL}

print('Downloaded:', PDF_PATH, os.path.getsize(PDF_PATH) // 1024, 'KB')


Downloaded: a320_manual.pdf 6726 KB


In [8]:
# TODO: Extract the text of the manual.
#
# The document is several hundred pages. You do not need all of it - Chapter 2
# (Aircraft Description) contains most of the specifications support executives
# are asked about.
#
# Steps:
#   1. Open the PDF with PdfReader(PDF_PATH)
#   2. Print how many pages it has
#   3. Loop over a range of pages and call .extract_text() on each
#   4. Join them into one string called manual_text
#
# Note: .extract_text() sometimes returns None for image-only pages. Guard against
#       that with  (page.extract_text() or '')
#
# Then print the character count and the first 1000 characters. PDF text extraction
# is messier than most people expect and you should look at it before trusting it.

PAGE_START = 15
PAGE_END = 90

manual_text = ''
reader = PdfReader(PDF_PATH)
print(f"Number of pages: {len(reader.pages)}")
pages_text = []
for i in range(PAGE_START,PAGE_END):
    page = reader.pages[i]
    pages_text.append(page.extract_text() or '')

manual_text = "\n".join(pages_text)
print('Characters extracted:', len(manual_text))
print(manual_text[:1000])


Number of pages: 427
Characters extracted: 54198
@A320
AIRCRAFT CHARACTERISTICS - AIRPORT AND MAINTENANCE PLANNING
L.E.C.
Page 14
Jun 01/24
CONTENT CHG
CODE
LAST REVISION
DATE
FIGURE Ground Service Connections - Green System Ground Service
Panel
May 01/16
FIGURE Ground Service Connections - Blue System Ground Service
Panel
May 01/16
FIGURE Ground Service Connections - Yellow System Ground Service
Panel
May 01/16
FIGURE Ground Service Connections - RAT
May 01/16
Subject 5-4-4
Electrical System
May 01/15
FIGURE Ground Service Connections - External Power Receptacles
May 01/14
Subject 5-4-5
Oxygen System
May 01/14
FIGURE Ground Service Connections - Oxygen System
May 01/14
Subject 5-4-6
Fuel System
May 01/14
FIGURE Ground Service Connections - Refuel/Defuel Control Panel
May 01/14
FIGURE Ground Service Connections - Refuel/Defuel Couplings
May 01/14
FIGURE Ground Service Connections - Overwing Gravity-Refuel Cap
(If Installed)
May 01/14
FIGURE Ground Service Connections - Overpressure Pro

In [9]:
# TODO: Write chunk_text(text, size, overlap).
#
# The function should cut `text` into consecutive pieces of `size` characters,
# where each piece repeats the last `overlap` characters of the previous one.
#
# Example with size=10, overlap=3 on 'ABCDEFGHIJKLMNOP':
#   chunk 1 -> 'ABCDEFGHIJ'
#   chunk 2 -> 'HIJKLMNOP'      (starts 3 characters back)
#
# Why overlap? A sentence sitting exactly on a boundary would otherwise be split in
# half, and neither half would answer the question. The overlap guarantees that
# sentences near a boundary appear complete in at least one chunk.
#
# Careful: if you advance the start position by (size - overlap) you must make sure
# that value is positive, or your loop will never end.

def chunk_text(text, size=5000, overlap=500):
    # your code here
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start = max(0, end - overlap)
    return chunks;

CHUNKS = chunk_text(manual_text, size=5000, overlap=500)
print('Number of chunks:', len(CHUNKS))


Number of chunks: 13


In [10]:
# TODO: Confirm the overlap is really there.
#
# Print the last 200 characters of CHUNKS[0] and the first 200 characters of
# CHUNKS[1]. The end of the first should reappear at the start of the second.

# your code here
print(CHUNKS[0][-200:])
print('----')
print(CHUNKS[1][:200])




FIGURE Engine Exhaust Velocities - Breakaway Power – IAE V2500
Series Engine
Dec 01/15
FIGURE Engine Exhaust Velocities - Breakaway Power 12% MTO –
CFM LEAP-1A Engine
Dec 01/15
FIGURE Engine Exhaust V
----
tures - Ground Idle Power – CFM
LEAP-1A Engine
Dec 01/15
FIGURE Engine Exhaust Temperatures - Ground Idle Power – PW
1100G Engine
Dec 01/15
Subject 6-1-3
Engine Exhaust Velocities Contours - Breakaway


### Question 1.1

What would go wrong if you set the chunk size to 50 characters? What would go wrong
if you set it to 200,000?


*If the overlap is 50 some sentences may be more than 50 characters and hence will be split and data may not be make sense in either chunks leading to loss of information. On the other hand if overlap is large like 200000, it will lead to lot of duplicacy causing performace and ambiguous results.*


---

# Part 2 - Embeddings and the Vector Store

## Objective

Convert text into numbers so that meaning can be compared mathematically.

Keyword search fails when the user and the manual use different words for the same
thing. A user asks about "legroom"; the manual says "seat pitch". Embeddings map
text to vectors where similar meanings land close together, so the match survives
the change of vocabulary.


In [11]:
# TODO: Complete embed_texts().
#
# The call you need is:
#   result = genai.embed_content(model=EMBEDDING_MODEL, content=text, task_type=task_type)
#   vector = result['embedding']
#
# About task_type: embedding models are trained to place a *question* near the
# *passage that answers it*, which is not the same as placing two passages near each
# other. Use 'retrieval_document' when embedding chunks and 'retrieval_query' when
# embedding a user question.
#
# Return a numpy array of shape (len(texts), embedding_dimension) with dtype float32.
#
# Be kind to the API: embed in a loop with a short time.sleep() between calls. When
# verbose is True, print progress every 20 chunks so you can watch the slow cell work.
#
# Optional but recommended: wrap the API call in a try/except and retry once or twice
# before giving up. A few hundred consecutive calls will usually hit one transient
# error, and you do not want to lose ten minutes of embedding to it.

def embed_texts(texts, task_type='retrieval_document', verbose=False):
    # your code here
    #embeddings =
    embeddings = []

    #if verbose:
    #    for i in range(0,len(texts),20):
    #        print(i)
    #        result = genai.embed_content(model=EMBEDDING_MODEL, content=texts[i:i+20], task_type=task_type)
    #        vector = result['embedding']
    #        embeddings.extend(vector)
    #        time.sleep(1)
    if len(texts) > 20:
        for i in range(0,len(texts),20):
            if verbose:
                print(i)
            result = genai.embed_content(model=EMBEDDING_MODEL, content=texts[i:i+20], task_type=task_type)
            vector = result['embedding']
            embeddings.extend(vector)
            time.sleep(1)
    else:
        if verbose:
              print(len(texts))
        result = genai.embed_content(model=EMBEDDING_MODEL, content=texts, task_type=task_type)
        vector = result['embedding']
        embeddings.extend(vector)
        time.sleep(1)

    return np.array(embeddings, dtype=np.float32)


# Test on a single short string
test_vec = embed_texts(['How wide is the A320?'], task_type='retrieval_query')
print('Shape:', test_vec.shape)
print('Dimensions per vector:', test_vec.shape[1])
print('First 8 numbers:', test_vec[0][:8])


Shape: (1, 3072)
Dimensions per vector: 3072
First 8 numbers: [-0.00767049 -0.00471567  0.00574446 -0.01403486 -0.0112345  -0.01261654
 -0.00158145 -0.00966548]


In [12]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

phrases = [
    'What is the wingspan of the aircraft?',
    'How wide is the plane from wingtip to wingtip?',
    'What is the baggage allowance for economy passengers?',
]

for i in range(len(phrases)):
    for j in range(i+1,len(phrases)):
        print(phrases[i],phrases[j],cosine_similarity(embed_texts([phrases[i]])[0],embed_texts([phrases[j]])[0]))

What is the wingspan of the aircraft? How wide is the plane from wingtip to wingtip? 0.9312273
What is the wingspan of the aircraft? What is the baggage allowance for economy passengers? 0.66263074
How wide is the plane from wingtip to wingtip? What is the baggage allowance for economy passengers? 0.6467943


In [13]:
# TODO: Embed every chunk in CHUNKS.
#
# This is the slow cell - a few hundred chunks means a few hundred API calls.
# Use task_type='retrieval_document' this time, and pass verbose so you can watch it.
#
# Store the result in CHUNK_EMBEDDINGS.

CHUNK_EMBEDDINGS = None
# your code here
CHUNK_EMBEDDINGS = embed_texts(CHUNKS, task_type='retrieval_document', verbose=True)

print('Embedded', CHUNK_EMBEDDINGS.shape[0], 'chunks into', CHUNK_EMBEDDINGS.shape[1], 'dimensions')


13
Embedded 13 chunks into 3072 dimensions


In [14]:
# TODO: Build the vector store with FAISS.
#
# FAISS does not have a "cosine similarity" index, but there is a standard trick:
# if you normalise every vector to length 1, the dot product between two vectors
# *is* their cosine similarity. So:
#
#   1. Copy the embeddings (faiss.normalize_L2 modifies the array in place, and you
#      do not want to corrupt CHUNK_EMBEDDINGS)
#   2. Call faiss.normalize_L2(copy)
#   3. Create faiss.IndexFlatIP(dimension)      # IP = inner product = dot product
#   4. index.add(copy)
#
# Wrap this in build_index(embeddings) so you can build a second index later with
# different chunk settings.

def build_index(embeddings):
    # your code here
    copy = embeddings.copy()
    faiss.normalize_L2(copy)
    index = faiss.IndexFlatIP(copy.shape[1])
    index.add(copy)
    return index



INDEX = build_index(CHUNK_EMBEDDINGS)
print('Vectors in index:', INDEX.ntotal)


Vectors in index: 13


---

# Part 3 - The Retrieval Engine

## Objective

Find the right page of the manual for a given question.

Note that nothing in this part involves a language model. Retrieval is a search
problem, and it either works or it does not, independently of what the chatbot
later does with the result. Debug it on its own.


In [15]:
def retrieve_manual(query, k=3, index=None, chunks=None):
    index = INDEX if index is None else index
    chunks = CHUNKS if chunks is None else chunks


    # 1. Embed the query, which returns a 2D array of shape (1, 3072).
    query_embedding = embed_texts([query], task_type='retrieval_query')

    # 2. Normalize the query embedding. faiss.normalize_L2 expects a 2D array.
    #    It modifies the array in-place.
    faiss.normalize_L2(query_embedding)

    # 3. Perform the search using the normalized 2D query embedding.
    #    scores will be (1, k), indices will be (1, k)
    scores, indices = index.search(query_embedding, k)

    # 4. Construct the list of (chunk_text, score) pairs based on the retrieved indices.
    retrieved_items = []
    # indices[0] gives the 1D array of k indices for the first (and only) query.
    # scores[0] gives the 1D array of k scores for the first (and only) query.
    for i in range(k):
        chunk_index = indices[0][i]
        score = scores[0][i]
        retrieved_items.append((chunks[chunk_index], score))

    # Return the list of (chunk_text, score) pairs and the raw indices (indices[0]).
    return retrieved_items, indices[0]



# Try it
retrieved_results, _ = retrieve_manual('What is the wingspan of the A320?')
for text, score in retrieved_results:
    print(round(score, 4), '|', text[:120].replace('\n', ' '))

0.7496 | ID CGDESCRIPTIONFLAP TRACK 3FLAP TRACK 2FLAP TRACK 4C2.892.782.759.489.129.02ABCN_AC_020300_1_0430101_01_00 Ground Clear
0.7494 |  CC 18.26 m³ (645 ft³) Usable Volume, Bulk CC 5.88 m³ (208 ft³) Water Volume, FWD CC 15.56 m³ (549 ft³) Water Volume, AF
0.7377 | .5771.8630.5681.9420.5921.8930.577N10.6724.0642.2041.2391.8930.5771.8630.5681.9420.5921.8930.577N10.8544.6452.8011.4162.


In [16]:
# TODO: Test the retriever on its own, before any language model is involved.
#
# For each query below, print the similarity score and the first ~300 characters of
# each retrieved chunk. Then read them and decide, for each query, whether the answer
# is actually in there.
#
# Pay close attention to the third query. The manual cannot possibly answer it.
# Look hard at what score comes back anyway.

test_queries = [
    'What is the wingspan of the A320?',
    'How much does the aircraft weigh when empty?',
    "What is the CEO's name?",
]

# your code here
for query in test_queries:
    print(query)
    retrieved_results, _ = retrieve_manual(query)
    for text, score in retrieved_results:
        print(round(score, 4), '|', text[:300].replace('\n', ' '))


What is the wingspan of the A320?
0.7496 | ID CGDESCRIPTIONFLAP TRACK 3FLAP TRACK 2FLAP TRACK 4C2.892.782.759.489.129.02ABCN_AC_020300_1_0430101_01_00 Ground Clearances Flap Tracks - 1 + F FIGURE-2-3-0-991-043-A01 @A320 AIRCRAFT CHARACTERISTICS - AIRPORT AND MAINTENANCE PLANNING 2-3-0 Page 13 Jun 01/24 **ON A/C A320-200 A320neo mftmftmftAB4.
0.7494 |  CC 18.26 m³ (645 ft³) Usable Volume, Bulk CC 5.88 m³ (208 ft³) Water Volume, FWD CC 15.56 m³ (549 ft³) Water Volume, AFT CC 20.77 m³ (733 ft³) Water Volume, Bulk CC 7.76 m³ (274 ft³)   @A320 AIRCRAFT CHARACTERISTICS - AIRPORT AND MAINTENANCE PLANNING 2-2-0 Page 1 Jun 01/24 2-2-0 General Aircraft Di
0.7377 | .5771.8630.5681.9420.5921.8930.577N10.6724.0642.2041.2391.8930.5771.8630.5681.9420.5921.8930.577N10.8544.6452.8011.4162.4800.7562.4600.7502.5220.7692.4900.759N1SHARKLETBOTTOMSHARKLET TOP ftmMRW (WV0)73 900 kg (162 922 lb)MRW (WV15)78 400 kg (172 842 lb)OEW41 244 kg(90 927 lb)6.79823.92722.3037.29321
How much does the aircraft weigh whe

### Question 3.1

For each of the three queries, does the retrieved text actually contain the answer?

What similarity scores came back for the third query, and what does that tell you
about a retriever that always returns something?


*No, only for second question it could be 62,800 Kg. But even this is not completely clear.*


In [17]:
# TODO: Tuning experiment.
#
# Rebuild the whole pipeline with smaller chunks - size 1000, overlap 100 - and run
# the same three queries against it.
#
# Steps:
#   1. small_chunks = chunk_text(manual_text, size=1000, overlap=100)
#   2. Embed them (this takes longer - there are more chunks now)
#   3. small_index = build_index(...)
#   4. Re-run the three test queries, passing index=small_index, chunks=small_chunks
#
# Print how many chunks each configuration produced, then compare the retrieved text.

# your code here
small_chunks = chunk_text(manual_text, size=1000, overlap=100)
small_index = build_index(embed_texts(small_chunks, task_type='retrieval_document'))
for query in test_queries:
    print(query)
    retrieved_results, _ = retrieve_manual(query, index=small_index, chunks=small_chunks)
    for text, score in retrieved_results:
        print(round(score, 4), '|', text[:300].replace('\n', ' '))



What is the wingspan of the A320?
0.7565 | 0300_1_0440101_01_00 Ground Clearances Aileron Up FIGURE-2-3-0-991-044-A01 @A320 AIRCRAFT CHARACTERISTICS - AIRPORT AND MAINTENANCE PLANNING 2-3-0 Page 15 Jun 01/24 **ON A/C A320-200 A320neo mftmftmftBC4.074.214.353.964.114.243.944.084.2113.3512.9912.9313.8114.2713.4813.9113.3913.81SPOILERS EXTENDED
0.7489 | -991-004-A01 @A320 AIRCRAFT CHARACTERISTICS - AIRPORT AND MAINTENANCE PLANNING 2-2-0 Page 5 Jun 01/24 **ON A/C A320-200 RELATED TO AIRCRAFT ATTITUDE AND WEIGHT.NOTE: 37.57 m(123.27 ft)5.75 m(18.86 ft)6.07 m(19.91 ft)3.95 m(12.96 ft)4.87 m(15.98 ft)1.24 m(4.07 ft)3.31 m(10.86 ft)8.30 m(27.23 ft)11.91
0.7455 |  CC 18.26 m³ (645 ft³) Usable Volume, Bulk CC 5.88 m³ (208 ft³) Water Volume, FWD CC 15.56 m³ (549 ft³) Water Volume, AFT CC 20.77 m³ (733 ft³) Water Volume, Bulk CC 7.76 m³ (274 ft³)   @A320 AIRCRAFT CHARACTERISTICS - AIRPORT AND MAINTENANCE PLANNING 2-2-0 Page 1 Jun 01/24 2-2-0 General Aircraft Di
How much does the aircraft weigh whe

### Question 3.2

Which configuration retrieved more precise passages? Which one gave the model more
surrounding context to work with? Which would you ship, and why?


*Smaller Chunks retrieved better passages. Smaller chunks gave more context. Smaller Chunks of 1000 and overlap of 100 is better to ship as it gave better results.*


### Exploration (not graded)

Try varying `k`. With `k=1` the model sees one passage and little else; with `k=10`
it sees a great deal of text, most of it irrelevant.

Does a larger `k` always produce better answers? What does it cost you in tokens and
in latency?


In [18]:
# Optional: how much text are you actually sending at each value of k?

for k in [1, 3, 5, 10]:
    # retrieve_manual returns (list_of_tuples, indices_array), so we take the first element.
    retrieved_results, _ = retrieve_manual('What is the wingspan of the A320?', k=k)
    total_chars = sum(len(t) for t, _ in retrieved_results)
    approx_tokens = total_chars // 4
    print('k={:<3} chunks={:<3} characters sent={:<7} approx tokens={}'.format(
        k, len(retrieved_results), total_chars, approx_tokens))

k=1   chunks=1   characters sent=5000    approx tokens=1250
k=3   chunks=3   characters sent=15000   approx tokens=3750
k=5   chunks=5   characters sent=25000   approx tokens=6250
k=10  chunks=10  characters sent=49698   approx tokens=12424


---

> ## Critical Incident Report
>
> The retrieval engine is working. On a test question about wingspan, the correct
> passage was returned as the top result - and the assistant **still** produced a
> number that does not appear anywhere in that passage.
>
> Retrieval alone is not grounding. The passage has to be placed in front of the
> model with instructions strict enough that the model prefers the document over its
> own memory, and honest enough that it says nothing when the document says nothing.


---

# Part 4 - Grounded Generation and Prompt Engineering

## Objective

Force the model to answer from the retrieved context, and only from it.


In [19]:
# Your Assignment 02 code, reproduced here so this notebook runs standalone.
#
# Nothing to do in this cell. If you improved these functions last time, paste your
# own versions in instead.

import requests as http_requests

OLLAMA_MODEL = 'gemma4:12b' # TODO: Add your own Ollama model
ACTIVE_MODEL = 'Gemini'          # 'Gemini' or 'Ollama'


def askGemini(prompt, system=None, temperature=0.3):
    full_prompt = prompt if system is None else system + '\n\n' + prompt
    config = genai.types.GenerationConfig(temperature=temperature)
    response = gemini_model.generate_content(full_prompt, generation_config=config)
    return response.text


def askLocalModel(prompt, system=None, model=OLLAMA_MODEL, temperature=None):
    payload = {'model': model, 'prompt': prompt, 'stream': False}
    if system:
        payload['system'] = system
    if temperature is not None:
        payload['options'] = {'temperature': temperature}
    r = http_requests.post('http://localhost:11434/api/generate', json=payload)
    return r.json()['response']


def ask(prompt, system=None, temperature=0.3):
    if ACTIVE_MODEL == 'Gemini':
        return askGemini(prompt, system=system, temperature=temperature)
    if ACTIVE_MODEL == 'Ollama':
        return askLocalModel(prompt, system=system, temperature=temperature)
    return '[No model selected]'


print('Model routing ready. Active model:', ACTIVE_MODEL)


Model routing ready. Active model: Gemini


In [20]:
# TODO: Write the two system prompts.
#
# GROUNDED_SYSTEM_PROMPT is used when retrieval is on. It needs three things:
#   - a role      ("You are an AeroWing Technical Operations Assistant.")
#   - a constraint (answer only from the provided context; if the answer is not
#                   there, reply exactly 'Data not available in manual.')
#   - a tone      (precise, concise, professional)
#
# PLAIN_SYSTEM_PROMPT is used when retrieval is off, so that Part 6 compares
# like with like. It should describe the same assistant *without* the grounding
# constraint - otherwise the ungrounded run refuses everything and the comparison
# tells you nothing.

GROUNDED_SYSTEM_PROMPT = (
    # your prompt here
    'You are an AeroWing Technical Operations Assistant. Answer only from the provided context; if the answer is not there, reply exactly "Data not available in manual". Be professional in your reply.'
)

PLAIN_SYSTEM_PROMPT = (
    # your prompt here
    'You are an AeroWing Technical Operations Assistant. Reply the question asked and be professional.'
)

print('System prompts defined.')


System prompts defined.


In [21]:
# TODO: Complete build_prompt(query, retrieved).
#
# `retrieved` is the list of (chunk_text, score) pairs from retrieve_manual.
#
# Assemble a single string in this shape:
#
#     Context Information:
#     [Source 1]
#     ...text of first chunk...
#
#     [Source 2]
#     ...text of second chunk...
#
#     User Question:
#     ...the query...
#
#     Instructions:
#     Answer the question based strictly on the context above.
#     If the context does not contain the answer, say so.
#
# Numbering the sources is not decoration - it lets the model tell you which passage
# it used, and lets you check.

def build_prompt(query, retrieved):
    # your code here
    prompt = 'Context Information:\n'
    for i, (text, score) in enumerate(retrieved):
        #print(i,score,'-----------------')
        prompt += f'[Source {i+1}]\n{text}\n'
    #print('----------------USER QUESTION---------------')
    prompt += '\nUser Question:\n'
    #print('----------------USER QUESTION---------------')
    prompt += f'{query}\n'
    #print('----------------USER QUESTION---------------')
    prompt += '\nInstructions:\nAnswer the question based strictly on the context above. If the context does not contain the answer, say so.'
    return prompt


# Inspect the prompt before you ever send it. Most RAG bugs are visible here.
retrieved_results, _ = retrieve_manual('What is the wingspan of the A320?')
#print(retrieved_results)
example = build_prompt('What is the wingspan of the A320?',retrieved_results)
print(example[:1500])


Context Information:
[Source 1]
ID CGDESCRIPTIONFLAP TRACK 3FLAP TRACK 2FLAP TRACK 4C2.892.782.759.489.129.02ABCN_AC_020300_1_0430101_01_00
Ground Clearances
Flap Tracks - 1 + F
FIGURE-2-3-0-991-043-A01
@A320
AIRCRAFT CHARACTERISTICS - AIRPORT AND MAINTENANCE PLANNING
2-3-0
Page 13
Jun 01/24
**ON A/C A320-200 A320neo
mftmftmftAB4.174.064.0113.6813.3213.163.833.723.6812.5712.2112.07AILERON DOWNMAXIMUM RAMPWEIGHT AFT CGMAXIMUM RAMPWEIGHT FWD CGA/C IN MAINTENANCECONFIGURATION MID CGDESCRIPTIONAILERON OUTBDAILERON INBDBA
N_AC_020300_1_0190101_01_01
Ground Clearances
Aileron Down
FIGURE-2-3-0-991-019-A01
@A320
AIRCRAFT CHARACTERISTICS - AIRPORT AND MAINTENANCE PLANNING
2-3-0
Page 14
Jun 01/24
**ON A/C A320-200 A320neo
mftmftmftAB4.554.444.3914.9314.5714.404.354.244.2014.2713.9113.78AILERON UPMAXIMUM RAMPWEIGHT AFT CGMAXIMUM RAMPWEIGHT FWD CGA/C IN MAINTENANCECONFIGURATION MID CGDESCRIPTIONAILERON OUTBDAILERON INBDBA
N_AC_020300_1_0440101_01_00
Ground Clearances
Aileron Up
FIGURE-2-3-0-991-0

In [22]:
# TODO: Complete answer_question().
#
# When use_rag is True:
#   retrieve, build the prompt, send it with GROUNDED_SYSTEM_PROMPT
# When use_rag is False:
#   send the bare query with PLAIN_SYSTEM_PROMPT
#
# Return BOTH the reply text and the retrieved chunks - Part 5 needs the chunks to
# display sources, and Part 6 needs them to check the answer against them.

def answer_question(query, k=3, use_rag=True):
    # your code here
    retrieved_results, indices = retrieve_manual(query, k=k)
    prompt = build_prompt(query, retrieved_results)
    if use_rag:
        reply = ask(prompt, system=GROUNDED_SYSTEM_PROMPT)
    else:
        reply = ask(prompt, system=PLAIN_SYSTEM_PROMPT)
    return reply, retrieved_results



reply, sources = answer_question('What is the wingspan of the A320?')
print(reply)
print()
print('Sources used:', len(sources))


The wingspan of the A320 is 34.10 m (111.88 ft) for the A320-200 (Wing Tip Fence configuration) and 35.80 m (117.45 ft) for the A320-200 (Sharklet configuration) and A320neo.

Sources used: 3


## Experiment - the same question, three ways

Send one factual question that the manual *can* answer under three conditions:

1. **No context at all** - the model answers from memory.
2. **Three random chunks** as context - the prompt contains manual text, but not the
   right manual text.
3. **The three retrieved chunks** as context.

Condition 2 is the one that matters. Without it you cannot tell whether your
improvement came from *retrieval* or merely from the presence of some official
looking text in the prompt. Skipping it is how teams end up shipping a pipeline
whose retriever does nothing.


In [23]:
# TODO: Run the three conditions and print all three answers.
#
# For condition 2, use random.sample(CHUNKS, 3) to pick chunks, and pass them to
# build_prompt in the same (text, score) format retrieve_manual returns - a score of
# 0.0 is fine as a placeholder.
#
# Set the random seed first so your result is reproducible.

random.seed(42)
question = 'What is the wingspan of the A320?'


# your code here
#Condition 1
print(question)
print('Condition 1')
reply, sources = answer_question(question, use_rag=False)
print(reply)
print()
print('Sources used:', len(sources))
print()
#Condition 2
print('Condition 2')
randomChunks = random.sample(CHUNKS, 3)
randomRetrieved = []
for chunk in randomChunks:
    randomRetrieved.append((chunk,0.0))
prompt = build_prompt(query, randomRetrieved)
reply = ask(prompt, system=GROUNDED_SYSTEM_PROMPT)
print(reply)
print()

#Condition 3
print('Condition 3')
reply, sources = answer_question(question, use_rag=True)
print(reply)
print()
print('Sources used:', len(sources))




What is the wingspan of the A320?
Condition 1
Based on the provided documentation, the wingspan of the A320 is as follows:

*   **A320-200 (Wing Tip Fence):** 34.10 m (111.88 ft)
*   **A320-200 (Sharklet):** 35.80 m (117.45 ft)
*   **A320neo:** 35.80 m (117.45 ft)

Sources used: 3

Condition 2
Data not available in manual

Condition 3
Based on the provided manual, the wingspan for the A320-200 is 34.10 m (111.88 ft) for the Wing Tip Fence configuration, and 35.80 m (117.45 ft) for the Sharklet configuration. For the A320neo, the wingspan is 35.80 m (117.45 ft).

Sources used: 3


### Question 4.1

Compare the three answers. Which condition produced the most trustworthy answer, and
how would you know it was trustworthy without already knowing the correct figure?


*Condition 3 is the most trust worthy as it clearly mentions the wingspan legnth for variuos variants unlike just dumping the complete information*


In [24]:
# TODO: The refusal test.
#
# Ask something the manual cannot possibly answer:
#   'What is the in-flight Wi-Fi password?'
#
# Run it through answer_question with use_rag=True.
#
# If your assistant invents an answer, your system prompt is too weak. Strengthen it
# and try again - and keep BOTH versions. Record the prompt that failed and the
# prompt that worked in the markdown cell below. That comparison is the deliverable,
# not just the final working version.

refusal_question = 'What is the in-flight Wi-Fi password?'

# your code here
reply, sources = answer_question(refusal_question, use_rag=True)
print(reply)
print()
print('Sources used:', len(sources))


Data not available in manual

Sources used: 3


### Question 4.2

Paste the system prompt that failed and the system prompt that worked. What was the
specific change that fixed it?


*The System prompt did not invent the answer in the first instance itself. That's beacause we have mentioend in grounded prompt to strictly check the context and mention that Data is not available if not found in context. The grounded prompt we used is : "You are an AeroWing Technical Operations Assistant. Answer only from the provided context; if the answer is not there, reply exactly "Data not available in manual". Be professional in your reply."*


---

# Part 5 - Deploying AeroBrain

## Objective

Wire retrieval into the chatbot you already built.

You are not starting from a blank notebook. This is your AeroAssist application from
Assignment 02 with three additions: a RAG on/off toggle, a source panel under each
answer, and a "thinking" indicator, because retrieval plus generation is noticeably
slower than generation alone.


In [25]:
# The page shell. Nothing to do here - the CSS and the element IDs are given.
#
# The IDs your JavaScript will need:
#   #chat-window   where messages are appended
#   #user-input    the text box
#   #send-btn      the send button
#   #rag-toggle    the checkbox that turns retrieval on and off
#   #thinking      the hidden "checking the manual" indicator

HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
  <title>AeroBrain</title>
  <style>
    body { font-family: system-ui, Arial, sans-serif; background:#eef1f5; margin:0; padding:24px; }
    #app { max-width:640px; margin:0 auto; background:#fff; border-radius:10px;
           padding:18px; box-shadow:0 2px 10px rgba(0,0,0,.08); }
    h3 { margin:0 0 4px 0; color:#1b497d; }
    .sub { color:#6b7280; font-size:13px; margin-bottom:12px; }
    #chat-window { height:380px; overflow-y:auto; border:1px solid #d7dbe0;
                   border-radius:6px; padding:12px; background:#fbfcfd; }
    .msg { margin-bottom:12px; line-height:1.45; }
    .msg b { color:#1b497d; }
    details { margin-top:6px; font-size:12px; background:#f1f3f7;
              border-radius:5px; padding:6px 8px; }
    details pre { white-space:pre-wrap; color:#374151; margin:6px 0 0 0; }
    #thinking { display:none; color:#6b7280; font-style:italic; padding:8px 2px; }
    #controls { display:flex; gap:8px; margin-top:12px; align-items:center; }
    #user-input { flex:1; padding:9px; border:1px solid #d7dbe0; border-radius:6px; }
    #send-btn { padding:9px 18px; border:0; border-radius:6px;
                background:#1b497d; color:#fff; cursor:pointer; }
    label { font-size:13px; color:#374151; }
  </style>
</head>
<body>
  <div id="app">
    <h3>AeroBrain</h3>
    <div class="sub">AeroWing Technical Operations Assistant</div>
    <div id="chat-window"></div>
    <div id="thinking">AeroBrain is checking the manual...</div>
    <div id="controls">
      <input id="user-input" placeholder="Ask about the A320..."
             onkeydown="if(event.key==='Enter'){sendMessage();}" />
      <button id="send-btn" onclick="sendMessage()">Send</button>
      <label><input type="checkbox" id="rag-toggle" checked /> Use manual</label>
    </div>
  </div>
"""

print('Page shell defined.')


Page shell defined.


In [26]:
# TODO: Fill in the three marked places in the JavaScript below.
#
# TODO 1 - render the sources.
#   `sources` arrives as a list of objects: { score: 0.83, text: "..." }
#   If the list is non-empty, append a <details> block to the message so the user can
#   expand it and see which passages the answer came from. Something like:
#     <details><summary>Sources (3)</summary><pre>...</pre></details>
#
# TODO 2 - read the checkbox.
#   document.getElementById('rag-toggle').checked
#
# TODO 3 - show and hide the thinking indicator.
#   Set style.display to 'block' before the fetch and back to 'none' when the
#   response arrives. Remember to hide it in .catch() too, or a failed request
#   leaves the interface stuck.

JS_SCRIPT = """
<script>

function addMessage(sender, text, sources) {
    const chatWindow = document.getElementById("chat-window");
    const div = document.createElement("div");
    div.className = "msg";
    div.innerHTML = "<b>" + sender + ":</b> " + text;

    // TODO 1: if sources is a non-empty array, append a <details> block here
    if (sources && sources.length > 0) {
        const details = document.createElement("details");
        const summary = document.createElement("summary");
        summary.textContent = `Sources (${sources.length})`;
        details.appendChild(summary);
        const pre = document.createElement("pre");
        pre.textContent = JSON.stringify(sources, null, 2);
        details.appendChild(pre);
        div.appendChild(details);
    }

    chatWindow.appendChild(div);
    chatWindow.scrollTop = chatWindow.scrollHeight;
}

function sendMessage() {
    const inputBox = document.getElementById("user-input");
    const message = inputBox.value.trim();
    if (!message) return;

    addMessage("You", message, null);
    inputBox.value = "";

    // TODO 2: read the RAG toggle instead of hardcoding true
    const useRag = document.getElementById("rag-toggle").checked;

    // TODO 3: show the thinking indicator
    document.getElementById("thinking").style.display = 'block';

    fetch("/chat", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ message: message, use_rag: useRag })
    })
    .then(response => response.json())
    .then(data => {
        // TODO 3: hide the thinking indicator
        document.getElementById("thinking").style.display = 'none';
        addMessage("AeroBrain", data.reply, data.sources);
    })
    .catch(err => {
        // TODO 3: hide the thinking indicator here too
        document.getElementById("thinking").style.display = 'none';
        addMessage("System", "Request failed: " + err, null);
    });
}

</script>
</body>
</html>
"""

FULL_PAGE = HTML_PAGE + JS_SCRIPT
print('Page assembled.')


Page assembled.


In [27]:
# TODO: Complete the /chat route.
#
# The route should:
#   1. Read `message` and `use_rag` from the request JSON
#   2. Call answer_question(message, use_rag=use_rag)
#   3. Return JSON with keys: reply, model, rag, sources
#
# `sources` must be JSON-serialisable, so convert the (text, score) pairs into
# dictionaries and truncate the text to a few hundred characters - nobody wants a
# 5000-character wall of manual in a popup.

app = Flask(__name__)
CORS(app)


@app.route('/')
def home():
    return FULL_PAGE


@app.route('/model')
def model_info():
    return jsonify({'model': ACTIVE_MODEL})


app.view_functions.pop('chat', None)
@app.route('/chat', methods=['POST'])
def chat():
    payload = request.json or {}
    message = payload.get('message', '')
    use_rag = bool(payload.get('use_rag', True))

    # TODO: replace these two placeholder lines
    reply = []
    sources = []

    # your code here
    reply, sources = answer_question(message, use_rag=use_rag)
    sources = [{'text': s[0][:500], 'score': float(s[1])} for s in sources]



    return jsonify({'reply': reply, 'model': ACTIVE_MODEL,
                    'rag': use_rag, 'sources': sources})


print('Routes defined.')


Routes defined.


In [28]:
# Launch the server.
#
# Open the printed URL in a new tab. To stop the server, interrupt this cell.

url = eval_js('google.colab.kernel.proxyPort(5000)')
print('AeroBrain is running at:', url)
print('Open the link above in a new tab.')

app.run(port=5000)


AeroBrain is running at: https://5000-m-s-kkb-use1c0-1yhwp90t43dy9-c.us-east1-0.prod.colab.dev
Open the link above in a new tab.
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


> **Before moving on:** take two screenshots and attach them to your submission zip.
>
> 1. `part5_grounded.png` - an answer with its source panel expanded.
> 2. `part5_refusal.png` - the assistant correctly refusing a question the manual
>    cannot answer.
>
> Try the same question with the **Use manual** box ticked and unticked. The
> difference is the entire point of this assignment.


---

# Part 6 - Evaluation

## Objective

Measure whether grounding actually worked.

An engineer does not claim a system is better; an engineer shows it.


In [49]:
# The test set: four questions the manual can answer, two it cannot.
#
# You may swap any of these for your own, but keep the 4/2 split - a test set with
# no unanswerable questions cannot detect hallucination at all, which is the specific
# failure this assignment is about.

EVAL_QUESTIONS = [
    {'q': 'What is the wingspan of the A320?', 'answerable': True},
    {'q': 'What is the overall length of the aircraft?', 'answerable': True},
    {'q': 'What is the maximum ramp weight of the A320?', 'answerable': True},
    {'q': 'What are the main landing gear tyre dimensions?', 'answerable': True},
    {'q': 'What is the refund policy for a cancelled AeroWing booking?', 'answerable': False},
    {'q': 'How many cabin crew are rostered on the Delhi to Dubai route?', 'answerable': False},
]

print(len(EVAL_QUESTIONS), 'questions;',
      sum(1 for x in EVAL_QUESTIONS if not x['answerable']), 'of them unanswerable')


6 questions; 2 of them unanswerable


In [50]:
# TODO: Run every question twice - once with RAG off, once with RAG on - and print
#       the two answers side by side.
#
# For the questions marked answerable=False, watch specifically for whether the
# ungrounded run invents a policy. That is the number you will report below.
#
# Store your results as you go if you want to build the table programmatically.

# your code here
for question in EVAL_QUESTIONS:
    print(question['q'])
    print('Ungrounded:')
    reply, sources = answer_question(question['q'], use_rag=False)
    print(reply)
    print()
    print('Grounded:')
    reply, sources = answer_question(question['q'], use_rag=True)
    print(reply)
    print()
    print()


What is the wingspan of the A320?
Ungrounded:
Based on the provided documentation, the wingspan of the A320 is 35.80 meters (117.45 feet) for the A320neo and 34.10 meters (111.88 feet) for the A320-200.

Grounded:
Based on the provided manual, the wingspan of the A320 is 34.10 m (111.88 ft) for the A320-200 (Wing Tip Fence configuration) and 35.80 m (117.45 ft) for the A320-200 (Sharklet configuration) and the A320neo.


What is the overall length of the aircraft?
Ungrounded:
Based on the provided context, the overall length of the A320-200 and A320neo aircraft is 37.57 m (123.27 ft).

Grounded:
Data not available in manual


What is the maximum ramp weight of the A320?
Ungrounded:
As an AeroWing Technical Operations Assistant, I have reviewed the provided documentation regarding A320 aircraft characteristics.

The Maximum Ramp Weight (MRW) for the A320 varies depending on the specific aircraft model and weight variant (WV). Based on the provided tables, the values are as follows:

**A

### Question 6.1 - Results table

Fill in the table below from your runs. Keep the answers short - a phrase or a
figure is enough. In the last column write **yes** only if the answer is supported by
a passage that was actually retrieved.

| Question | Answer without RAG | Answer with RAG | Grounded? |
| --- | --- | --- | --- |
| Wingspan | 34.1,35.8 | 34.1,35.8 | yes |
| Overall length | 35.57 | NA | no |
| Maximum ramp weight | 73900Kg,75900Kg | 73900Kg,68400Kg | yes |
| Landing gear tyres | NA  | NA | no |
| Refund policy | NA | NA | no |
| Cabin crew roster | NA | NA | no |


### Question 6.2

For how many of the six questions did the ungrounded assistant produce a confident
but unverifiable answer? How many did the grounded assistant produce? State the
numbers plainly.


**UnGrounded** : 2, Cannot verify for Overall length, Maximum Ramp Weight in some variants.
**Grounded**   : 3 confident answers.

### Question 6.3 - Find a case where RAG loses

Every honest evaluation has one. Find at least one question your system answers
**worse** with retrieval enabled, and explain what went wrong.

Some places to look: very general questions ("what kind of aircraft is this?"),
questions whose answer is split across a chunk boundary, or questions where a
confidently retrieved but irrelevant table crowds out what the model already knew.


*What type of aeroplane is A320?*


---

# Final Reflection

Answer each question in three to five sentences. Support your answers with
observations from your own experiments, not general statements about RAG.

---

**1.** In Assignment 02 you improved answers by pasting a quick-reference note
directly into the prompt. In this assignment you built a retrieval pipeline instead.
Describe one situation where the simpler approach is still the better engineering
choice.


---

**2.** Your retriever always returns its top *k* chunks, even for a question the
manual never addresses. Explain why this is a risk in a production system, and
describe one concrete way you could reduce it.

---

**3.** AeroWing wants to add the cabin crew handbook, the maintenance manual and the
refund policy to the same assistant. What would you need to change in your design,
and what new failure modes would you expect once several documents share one index?


---

**4.** Which had the greater effect on answer quality in your experiments: improving
the retrieval (chunk size, *k*) or improving the prompt? Support your answer with
specific observations from Part 3 and Part 4.



------



1. In Assignment02 when the quick reference text is less we passed it with
prompt to know baggage allowance. Its easier and quick. In such cases where the reference mataterial is short we can use the approach of sending it with prompt than using RAG.
2. In production system this can cause performance and security issues as irrelavant data is returned. We can have a threshold score to return chunks.
3. We can categorize the chunks based on the file. We need to provide relavant answer like the "The refund policy does not mention this scenario, etc" rather than no data available. Also we should not refer to text in Cabin Crew manual for a refund policy question. It could lead to security issue.
4. Improving the prompt had greater impact. Though we changed the chunk size from 5000 to 1000, it still did not provide proper answer for wing size. Changing k also did not help. A proper grounded prompt has gave the correct answer.




---

# Submission Guidelines

Submit a single zip file named `AeroBrain_YourName.zip` containing:

- this notebook, with all code cells completed and all questions answered
- `part5_grounded.png` - a screenshot of an answer with its sources shown
- `part5_refusal.png` - a screenshot of the assistant correctly refusing

Before you submit:

- Restart the kernel and run the notebook from top to bottom. It must complete
  without errors.
- Check that your Gemini API key is **not** hardcoded anywhere. Use Colab Secrets.
- Check that your reflection answers refer to your own results, not to RAG in
  general.

---

*A language model that has read everything still knows nothing about your airline.
Retrieval gives it a source. Good prompting teaches it to stay there.*
